# 《Attention Is All You Need》中译英 (Zh → En) Colab 一键训练台

**本笔记本已内置你的全部源码，无需再额外上传任何 `.zip` 文件！**

1. 顶部菜单 **代码执行程序 (Runtime) → 更改运行时类型 (Change runtime type)** → 选 **L4 GPU**（省额度首选）或 **A100 GPU**（最快）。
2. 直接从上到下依次运行下方 5 个单元格即可（数据和模型权重会自动存进你的 Google Drive，断线零损失）。

In [ ]:
# 1. 检查 GPU 型号并挂载 Google Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 自动释放内置源码包（无需手动上传 zip！若手动上传了新 zip 则优先使用新 zip）并安装依赖
import base64, io, zipfile
from pathlib import Path

EMBEDDED_CODE_B64 = (
    '...'
)

work_dir = Path('/content/translator')
work_dir.mkdir(parents=True, exist_ok=True)
manual_zip = Path('/content/translator_colab_code.zip')
if manual_zip.exists():
    with zipfile.ZipFile(manual_zip, 'r') as zf:
        zf.extractall(work_dir)
    print(f'已从手动上传的 {manual_zip} 解压代码')
else:
    raw_bytes = base64.b64decode(EMBEDDED_CODE_B64)
    with zipfile.ZipFile(io.BytesIO(raw_bytes), 'r') as zf:
        zf.extractall(work_dir)
    print(f'已自动释放笔记本内置源码至 {work_dir}（零额外上传！）')

%cd /content/translator
!pip install -q sentencepiece datasets sacrebleu pyyaml tqdm pytest
!pytest -q

In [ ]:
# 3. 选择训练档位并准备数据 (自动缓存在你的 Google Drive，只跑一次)
#   - 'colab_base' : 100 万句, 论文标准 6 层 512 维 Transformer Base (25000 步: T4 约 5.5~6 小时, L4 约 1.5 小时, A100 约 45 分钟)
#   - 'local_small': 6 万句短句, 4 层 256 维 (T4 约 15 分钟热身)
MODE = 'colab_base'

CONFIG_PATH = f'configs/{MODE}.yaml'
DRIVE_ROOT = f'/content/drive/MyDrive/translator_runs/{MODE}'
DATA_DIR = f'{DRIVE_ROOT}/data'
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints'

from pathlib import Path
if not (Path(DATA_DIR) / 'dataset.pt').exists():
    !python scripts/prepare_data.py --config {CONFIG_PATH} --out-dir {DATA_DIR}
else:
    print(f'检测到 Google Drive 已有预处理数据缓存: {DATA_DIR}/dataset.pt，直接秒读！')

In [ ]:
# 4. 启动训练（若 Drive 中已有 last.pt 则自动无损断点续训；T4 自动 fp16、L4/A100 自动 bf16+compile）
from pathlib import Path

last_ckpt = Path(CKPT_DIR) / 'last.pt'
if last_ckpt.exists():
    print(f'发现已有进度 {last_ckpt}，自动断点续训...')
    !python train.py --config {CONFIG_PATH} --data-dir {DATA_DIR} --save-dir {CKPT_DIR} --device cuda --resume {last_ckpt}
else:
    !python train.py --config {CONFIG_PATH} --data-dir {DATA_DIR} --save-dir {CKPT_DIR} --device cuda

In [ ]:
# 5. 训完收官：平均最近 5 个权重 -> 评测测试集 Beam Search SacreBLEU -> 永久存入 Google Drive -> 自动关机停计费！
avg_ckpt = f'{CKPT_DIR}/avg5_final.pt'
!python translate.py --ckpt-dir {CKPT_DIR} --save-avg {avg_ckpt} --device cuda --beam 4 --max-samples 500

print(f'\n全部完成！最终模型已永久保存在 Google Drive: {CKPT_DIR}/avg5_final.pt')
print('正在自动释放 GPU 运行时并停止计费...')
from google.colab import runtime
runtime.unassign()